# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements of psychological symptoms along with scores from assessments such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All record sets, fields, and columns are referenced by their `@id`.

Let's list the record sets in the dataset, then enumerate fields (columns) for each record set.

In [ ]:
# List all RecordSets (@id) in dataset
record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in metadata.recordSet]
print("Available RecordSets in dataset (by @id):")
for rs_id in record_sets:
    print(f"- {rs_id}")

# For each RecordSet, print associated fields (@id)
for rs_id in record_sets:
    rs_obj = [rs for rs in metadata.recordSet if (rs['@id'] if isinstance(rs, dict) else rs) == rs_id][0]
    print(f"\nRecordSet: {rs_id}")
    fields = rs_obj.get('field', [])
    if fields:
        print("  Fields (@id):")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    - {field_id}")
    else:
        print("  No fields found.")# For demonstration, show a couple records from the first record set
if record_sets:
    example_rs = record_sets[0]
    print(f"\nExample records from RecordSet {example_rs}:")
    for ix, record in enumerate(dataset.records(record_set=example_rs)):
        pprint.pprint(record)
        if ix >= 2:
            break

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis.

Here, we use the record set and field `@id`s found above.

In [ ]:
# Extract data from each record set to Pandas DataFrames referenced by their @id
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print available columns in the first record set
first_rs_id = record_sets[0] if record_sets else None
if first_rs_id:
    print(f"Columns in RecordSet {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Filter rows using a numeric field (e.g., GAD-7 score)
- Normalize that field
- Group by a demographic column (e.g., gender)

Reference each variable by its `@id`.

In [ ]:
# Choose the main record set and field IDs
# (Replace below as appropriate based on previous overview. Example: record set ID and field IDs from survey data.)
main_record_set_id = first_rs_id  # For demo, use the first record set
df = dataframes[main_record_set_id]

# Find available numeric fields
numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or all(pd.to_numeric(df[col], errors='coerce').notnull())]
print(f"Numeric fields in RecordSet {main_record_set_id}: {numeric_fields}")

# Example: Use 'gad7_score' (GAD-7 total score). Reference by its column name/@id.
# Replace below with exact @id as necessary
numeric_field_id = None
for nf in numeric_fields:
    if 'gad' in nf.lower() or 'gad7' in nf.lower():
        numeric_field_id = nf
        break
if numeric_field_id is None and numeric_fields:
    numeric_field_id = numeric_fields[0]

# Set threshold for filtering
threshold = 10
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group field: try 'gender' or similar demographic field
group_field_id = None
for col in df.columns:
    if 'gender' in col.lower():
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by {group_field_id}: Mean {numeric_field_id}")
    print(grouped_df.head())
else:
    print("No demographic grouping field (like gender) found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot the distribution of the numeric field and show group means if grouping worked.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouped_df was created, plot group means
if group_field_id:
    grouped_df.plot.bar(y=numeric_field_id, legend=False)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed survey-based mental health indicators.
- Numeric scores (such as GAD-7) can be filtered, normalized, and grouped for demographic analysis.
- The data structure makes use of explicit `@id` referencing, supporting reproducible and FAIR analyses.
- Further exploration can be done by evaluating other psychological assessment scores or demographic effects.